# Step 2B - Base Self-Consistency on `dev_tune_200`

Step 2A cleared the fragile runtime path. This notebook runs the real base/self-consistency preflight required before bridge comparisons.

It uses only:

```text
runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl
```

It does not touch `analysis_500`, `ablation_500`, `final_test_500`, or `final_test_full`.

Goal:

```text
1. Run frozen base LLaDA on dev_tune_200 with seed 0.
2. Run frozen base LLaDA on dev_tune_200 with seed 1.
3. Estimate SelfLoc_base from agreement between the two base runs on locality buckets.
```

This is still not a bridge comparison.

In [ ]:
%%bash
set -euo pipefail

# Colab usually already provides CUDA-compatible torch, so do not reinstall
# unpinned torch unless the environment is missing it.
pip install -q \
  "transformers==4.46.3" \
  "datasets>=4.0.0" \
  "accelerate>=1.11.0" \
  "sentencepiece>=0.2.0" \
  "packaging" \
  "bitsandbytes>=0.43.0"

In [ ]:
import subprocess
import torch, transformers, datasets, accelerate

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)

assert torch.cuda.is_available(), "This notebook needs a GPU runtime."
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Save Runtime Environment

In [ ]:
import json
from pathlib import Path
import torch, transformers, datasets, accelerate

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
assert project.exists(), f"Project directory missing after Drive mount: {project}"

env = {
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "accelerate": accelerate.__version__,
}
out = project / "runs/counterfact_direction1_v1/runtime_environment_step2b.json"
out.parent.mkdir(parents=True, exist_ok=True)
with out.open("w", encoding="utf-8") as f:
    json.dump(env, f, indent=2)

print("Wrote:", out)
print(json.dumps(env, indent=2))

## Preflight: Project and Protocol Files

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_DIR = Path("/content/drive/MyDrive/Masters/SB V2/SB")
assert PROJECT_DIR.exists(), f"Missing project dir: {PROJECT_DIR}"

required_files = [
    "llada_runtime_editor_eval.py",
    "llada_counterfact_protocol.py",
    "paper_protocol.md",
    "implementation_spec.md",
    "runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl",
    "runs/counterfact_direction1_v1/protocol/protocol_manifest.json",
    "runs/counterfact_direction1_v1/dev_tune_200_base_coverage/candidate_coverage.jsonl",
]

for name in required_files:
    file_path = PROJECT_DIR / name
    assert file_path.exists(), f"Missing required file: {file_path}"

try:
    commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=PROJECT_DIR,
        text=True,
    ).strip()
    print("git commit:", commit)
except Exception as exc:
    print("git commit unavailable:", repr(exc))

with (PROJECT_DIR / "runs/counterfact_direction1_v1/protocol/protocol_manifest.json").open() as f:
    manifest = json.load(f)

assert manifest["protocol_version"] == "counterfact_direction1_v1"
assert manifest["artifacts"]["dev_tune_200"]["summary"]["count"] == 200

dev_path = PROJECT_DIR / "runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl"
dev_rows = [json.loads(line) for line in dev_path.open() if line.strip()]
dev_case_ids = {str(r["case_id"]) for r in dev_rows}

coverage_path = PROJECT_DIR / "runs/counterfact_direction1_v1/dev_tune_200_base_coverage/candidate_coverage.jsonl"
coverage_rows = [json.loads(line) for line in coverage_path.open()]
coverage_case_ids = {str(r["case_id"]) for r in coverage_rows}
assert len(coverage_rows) == 200, f"Expected 200 G0 coverage rows, got {len(coverage_rows)}"
assert coverage_case_ids == dev_case_ids, (
    "G0 coverage case IDs do not match dev_tune_200. "
    f"Missing coverage: {sorted(dev_case_ids - coverage_case_ids)[:10]}, "
    f"Extra coverage: {sorted(coverage_case_ids - dev_case_ids)[:10]}"
)
assert all(bool(r["all_target_new_tokens_after_candidate_insert"]) for r in coverage_rows), \
    "G0 coverage artifact says candidate insertion failed for at least one edit."

print("Python:", sys.version)
print("Project dir OK:", PROJECT_DIR)
print("Protocol OK:", manifest["protocol_version"])
print("G0 coverage OK:", coverage_path)

## Overwrite Guards

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
run_dirs = [
    project / "runs/counterfact_direction1_v1/dev_tune_200_base_seed0",
    project / "runs/counterfact_direction1_v1/dev_tune_200_base_seed1",
    project / "runs/counterfact_direction1_v1/dev_tune_200_base_selfconsistency_report",
]

for d in run_dirs:
    if d.exists():
        raise RuntimeError(
            f"Run directory already exists: {d}. "
            "Delete it manually or create a new run name. Do not overwrite silently."
        )

print("Overwrite guard passed.")

## Step 2B.1: Base Run, Seed 0

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_base_seed0

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_base_seed0 \
  --methods base \
  --eval_samples 4 \
  --steps 4 \
  --bridge_topk 4 \
  --mc_rollouts 0 \
  --seed 0 \
  --use_4bit 1 \
  --dtype float16 \
  --skip_candidate_coverage 1 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_base_seed0/stdout.log

## Step 2B.2: Base Run, Seed 1

In [ ]:
%%bash
set -euo pipefail

cd "/content/drive/MyDrive/Masters/SB V2/SB"
mkdir -p runs/counterfact_direction1_v1/dev_tune_200_base_seed1

python -u llada_runtime_editor_eval.py \
  --model_id GSAI-ML/LLaDA-8B-Base \
  --edits_path runs/counterfact_direction1_v1/protocol/dev_tune_200.jsonl \
  --output_dir runs/counterfact_direction1_v1/dev_tune_200_base_seed1 \
  --methods base \
  --eval_samples 4 \
  --steps 4 \
  --bridge_topk 4 \
  --mc_rollouts 0 \
  --seed 1 \
  --use_4bit 1 \
  --dtype float16 \
  --skip_candidate_coverage 1 \
  2>&1 | tee runs/counterfact_direction1_v1/dev_tune_200_base_seed1/stdout.log

## Optional Prompt-Row Diagnostic

Run this after both seed runs finish and before building the report. It verifies whether repeated `(case_id, bucket)` groups exist and shows which prompt-level fields are available for alignment.


In [ ]:
import json
from pathlib import Path
from collections import Counter

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")

for name in ["dev_tune_200_base_seed0", "dev_tune_200_base_seed1"]:
    p = project / f"runs/counterfact_direction1_v1/{name}/per_case_results.jsonl"
    rows = [json.loads(line) for line in p.open() if line.strip()]
    unit_field = "edit_id" if rows and "edit_id" in rows[0] else "case_id"

    print()
    print(name)
    print("num rows:", len(rows))
    print("unit field:", unit_field)
    print("first row keys:", sorted(rows[0].keys()))
    print("bucket counts:", Counter(r["bucket"] for r in rows))

    case_dupes = Counter((str(r["case_id"]), r["bucket"]) for r in rows)
    case_dupes = [(k, v) for k, v in case_dupes.items() if v > 1]
    edit_dupes = Counter((str(r.get(unit_field, r["case_id"])), r["bucket"]) for r in rows)
    edit_dupes = [(k, v) for k, v in edit_dupes.items() if v > 1]

    print("duplicated (case_id, bucket) groups:", len(case_dupes))
    print("example case duplicates:", case_dupes[:5])
    print("duplicated (edit_id, bucket) groups:", len(edit_dupes))
    print("example edit duplicates:", edit_dupes[:5])


## Step 2B.3: Validate Run Configs and Build Self-Consistency Report

In [ ]:
import json
from pathlib import Path
from collections import defaultdict, Counter
import statistics
import hashlib

project = Path("/content/drive/MyDrive/Masters/SB V2/SB")
base0 = project / "runs/counterfact_direction1_v1/dev_tune_200_base_seed0"
base1 = project / "runs/counterfact_direction1_v1/dev_tune_200_base_seed1"
out_dir = project / "runs/counterfact_direction1_v1/dev_tune_200_base_selfconsistency_report"
out_dir.mkdir(parents=True, exist_ok=True)


def load_jsonl(path):
    return [json.loads(line) for line in path.open() if line.strip()]


def load_run_config(run_dir):
    path = run_dir / "run_config.json"
    assert path.exists(), f"Missing run config: {path}"
    with path.open() as f:
        return json.load(f)


def assert_common_base_config(cfg):
    expected = {
        "protocol_version": "counterfact_direction1_v1",
        "edit_access": "given_at_edit_time",
        "training_access": "none",
        "hyperparameter_access": "dev_tune_only",
        "model_id": "GSAI-ML/LLaDA-8B-Base",
        "eval_samples": 4,
        "methods": ["base"],
        "split_role": "dev_tune_200",
    }
    for key, value in expected.items():
        assert cfg.get(key) == value, f"Expected {key}={value!r}, got {cfg.get(key)!r}"

    rollout = cfg.get("rollout_config", {})
    rollout_expected = {
        "steps": 4,
        "bridge_topk": 4,
        "mc_rollouts": 0,
    }
    for key, value in rollout_expected.items():
        assert rollout.get(key) == value, f"Expected rollout_config.{key}={value!r}, got {rollout.get(key)!r}"

    cfg_text = json.dumps(cfg, sort_keys=True)
    assert "analysis_500" not in cfg_text
    assert "final_test_500" not in cfg_text
    assert "final_test_full" not in cfg_text


def comparable_config(cfg):
    ignore_keys = {"seed", "timestamp_utc"}
    comparable = {k: v for k, v in cfg.items() if k not in ignore_keys}
    # output path is not currently top-level, but keep this robust if added later.
    comparable.pop("output_dir", None)
    comparable.pop("run_name", None)
    comparable.pop("created_at", None)
    return comparable


def assert_protocol_config(run_dir, expected_seed):
    cfg = load_run_config(run_dir)
    assert_common_base_config(cfg)
    assert cfg.get("seed") == expected_seed, f"{run_dir}: wrong seed {cfg.get('seed')}"
    print("Protocol config OK:", run_dir)
    return cfg


cfg0 = assert_protocol_config(base0, 0)
cfg1 = assert_protocol_config(base1, 1)
assert comparable_config(cfg0) == comparable_config(cfg1), "Seed runs differ in more than seed/timestamp metadata"

rows0 = load_jsonl(base0 / "per_case_results.jsonl")
rows1 = load_jsonl(base1 / "per_case_results.jsonl")
assert rows0, "seed0 has no per-case rows"
assert rows1, "seed1 has no per-case rows"
assert len(rows0) == len(rows1), (len(rows0), len(rows1))

UNIT_FIELD = "edit_id" if "edit_id" in rows0[0] else "case_id"
print("Edit-balanced unit field:", UNIT_FIELD)


def unit_id(row):
    return str(row.get(UNIT_FIELD, row["case_id"]))


def stable_prompt_key(row, occurrence_index):
    """
    Build a prompt-level key under the edit-level unit.

    Primary unit:
      edit_id, if available.

    Prompt identity:
      prompt hash, plus case_id when available.

    Last fallback:
      occurrence index within (edit_id, bucket).
    """
    edit_unit = unit_id(row)
    bucket = str(row["bucket"])
    case_id = str(row.get("case_id", ""))

    prompt_text = None
    for field in ["rendered_prompt", "prompt_text", "prompt", "input_text"]:
        if field in row and row[field] is not None:
            prompt_text = str(row[field])
            break

    if prompt_text is not None:
        digest = hashlib.sha1(prompt_text.encode("utf-8")).hexdigest()[:16]
        return (edit_unit, bucket, case_id, f"prompt_sha1:{digest}")

    for field in ["prompt_uid", "prompt_id", "prompt_index", "prompt_idx"]:
        if field in row and row[field] is not None:
            return (edit_unit, bucket, case_id, f"{field}:{row[field]}")

    return (edit_unit, bucket, case_id, f"occurrence:{occurrence_index}")


def index_prompt_rows(rows):
    """
    Index every prompt-level row without collapsing repeated
    (edit_id, bucket) groups.
    """
    occurrence_counts = defaultdict(int)
    duplicate_counts = defaultdict(int)
    indexed = {}

    for row in rows:
        edit_unit = unit_id(row)
        bucket = str(row["bucket"])
        base = (edit_unit, bucket)

        occurrence_index = occurrence_counts[base]
        occurrence_counts[base] += 1

        key = stable_prompt_key(row, occurrence_index)

        # Protect against duplicate prompt text / duplicate prompt ids.
        if key in indexed:
            duplicate_counts[key] += 1
            key = (*key, f"duplicate:{duplicate_counts[key]}")
        else:
            duplicate_counts[key] = 0

        indexed[key] = row

    assert len(indexed) == len(rows), (
        f"Index collapsed rows: indexed={len(indexed)}, rows={len(rows)}"
    )

    return indexed


by_key0 = index_prompt_rows(rows0)
by_key1 = index_prompt_rows(rows1)

keys0 = set(by_key0)
keys1 = set(by_key1)

missing_in_seed1 = sorted(keys0 - keys1)
missing_in_seed0 = sorted(keys1 - keys0)

assert not missing_in_seed1, f"Rows missing in seed1: {missing_in_seed1[:10]}"
assert not missing_in_seed0, f"Rows missing in seed0: {missing_in_seed0[:10]}"

edit_bucket_dupes0 = Counter((unit_id(r), str(r["bucket"])) for r in rows0)
edit_bucket_dupes1 = Counter((unit_id(r), str(r["bucket"])) for r in rows1)

print(
    "Seed0 duplicated (edit_id, bucket) groups:",
    sum(1 for v in edit_bucket_dupes0.values() if v > 1),
)
print(
    "Seed1 duplicated (edit_id, bucket) groups:",
    sum(1 for v in edit_bucket_dupes1.values() if v > 1),
)

expected_buckets = {
    "rewrite",
    "declarative_paraphrases",
    "near_locality",
    "far_locality",
}
observed_buckets = {row["bucket"] for row in rows0} | {row["bucket"] for row in rows1}
missing_buckets = expected_buckets - observed_buckets
assert not missing_buckets, f"Missing expected buckets: {missing_buckets}"


def maybe_mean_prompt_weighted(rows, key):
    vals = [float(r[key]) for r in rows if key in r and r[key] is not None]
    return sum(vals) / len(vals) if vals else None


def maybe_mean_edit_balanced(rows, key):
    grouped = defaultdict(list)

    for row in rows:
        if key in row and row[key] is not None:
            grouped[unit_id(row)].append(float(row[key]))

    if not grouped:
        return None

    edit_means = [
        sum(values) / len(values)
        for values in grouped.values()
        if values
    ]

    return sum(edit_means) / len(edit_means) if edit_means else None


def aggregate(rows):
    grouped = defaultdict(list)

    for row in rows:
        grouped[row["bucket"]].append(row)

    summary = {}

    for bucket, group in grouped.items():
        edit_ids = {unit_id(row) for row in group}

        summary[bucket] = {
            "num_edits": len(edit_ids),
            "num_prompt_rows": len(group),

            "mean_exact_rate_prompt_weighted": maybe_mean_prompt_weighted(group, "exact_rate"),
            "mean_exact_rate_edit_balanced": maybe_mean_edit_balanced(group, "exact_rate"),

            "mean_greedy_exact_prompt_weighted": maybe_mean_prompt_weighted(group, "greedy_exact"),
            "mean_greedy_exact_edit_balanced": maybe_mean_edit_balanced(group, "greedy_exact"),

            "mean_token_f1_prompt_weighted": maybe_mean_prompt_weighted(group, "token_f1"),
            "mean_token_f1_edit_balanced": maybe_mean_edit_balanced(group, "token_f1"),

            "mean_malformed_rate_prompt_weighted": maybe_mean_prompt_weighted(group, "malformed_rate"),
            "mean_malformed_rate_edit_balanced": maybe_mean_edit_balanced(group, "malformed_rate"),

            "mean_target_false_positive_rate_prompt_weighted": maybe_mean_prompt_weighted(group, "target_false_positive_rate"),
            "mean_target_false_positive_rate_edit_balanced": maybe_mean_edit_balanced(group, "target_false_positive_rate"),

            "mean_sparse_guidance_kl_prompt_weighted": maybe_mean_prompt_weighted(group, "sparse_guidance_kl"),
            "mean_sparse_guidance_kl_edit_balanced": maybe_mean_edit_balanced(group, "sparse_guidance_kl"),

            "mean_base_margin_prompt_weighted": maybe_mean_prompt_weighted(group, "base_margin"),
            "mean_base_margin_edit_balanced": maybe_mean_edit_balanced(group, "base_margin"),
        }

    return summary


summary0 = aggregate(rows0)
summary1 = aggregate(rows1)

assert summary0["rewrite"]["num_edits"] == 200
assert summary1["rewrite"]["num_edits"] == 200
assert summary0["near_locality"]["num_edits"] == 200
assert summary1["near_locality"]["num_edits"] == 200
assert summary0["far_locality"]["num_edits"] == 200
assert summary1["far_locality"]["num_edits"] == 200

print()
print("Seed 0 base metrics:")
for bucket, summary in sorted(summary0.items()):
    print(bucket, summary)

print()
print("Seed 1 base metrics:")
for bucket, summary in sorted(summary1.items()):
    print(bucket, summary)


def normalize_text(value):
    return " ".join(str(value).strip().lower().split())


def sample_agreement_paired(row0, row1):
    outs0 = [normalize_text(x) for x in row0.get("sample_outputs", [])]
    outs1 = [normalize_text(x) for x in row1.get("sample_outputs", [])]
    n = min(len(outs0), len(outs1))
    assert n > 0, "Missing sample outputs"
    return sum(float(outs0[i] == outs1[i]) for i in range(n)) / n


def sample_agreement_all_pairs(row0, row1):
    outs0 = [normalize_text(x) for x in row0.get("sample_outputs", [])]
    outs1 = [normalize_text(x) for x in row1.get("sample_outputs", [])]
    assert outs0 and outs1, "Missing sample outputs"
    pairs = [float(left == right) for left in outs0 for right in outs1]
    return sum(pairs) / len(pairs)


def greedy_agreement(row0, row1):
    g0 = normalize_text(row0.get("greedy_output", ""))
    g1 = normalize_text(row1.get("greedy_output", ""))
    if not g0 and not g1:
        return None
    return float(g0 == g1)


main_selfloc_key = "sample_self_agreement_paired_edit_balanced"


def compute_selfloc_by_bucket(by_key0, by_key1, locality_buckets):
    result = {}

    shared_all = set(by_key0) & set(by_key1)

    for bucket in locality_buckets:
        shared = sorted(
            key for key in shared_all
            if key[1] == bucket
        )

        assert shared, f"No shared prompt rows for bucket={bucket}"

        edit_to_paired = defaultdict(list)
        edit_to_all_pairs = defaultdict(list)
        edit_to_greedy = defaultdict(list)

        for key in shared:
            row0 = by_key0[key]
            row1 = by_key1[key]
            edit_unit = key[0]

            edit_to_paired[edit_unit].append(
                sample_agreement_paired(row0, row1)
            )
            edit_to_all_pairs[edit_unit].append(
                sample_agreement_all_pairs(row0, row1)
            )

            greedy_score = greedy_agreement(row0, row1)
            if greedy_score is not None:
                edit_to_greedy[edit_unit].append(greedy_score)

        paired_edit_means = [
            statistics.mean(values)
            for values in edit_to_paired.values()
            if values
        ]

        all_pair_edit_means = [
            statistics.mean(values)
            for values in edit_to_all_pairs.values()
            if values
        ]

        greedy_edit_means = [
            statistics.mean(values)
            for values in edit_to_greedy.values()
            if values
        ]

        prompt_weighted_paired = [
            score
            for values in edit_to_paired.values()
            for score in values
        ]

        prompt_weighted_all_pairs = [
            score
            for values in edit_to_all_pairs.values()
            for score in values
        ]

        result[bucket] = {
            "num_edits": len(edit_to_paired),
            "num_prompt_rows": len(shared),

            "sample_self_agreement_paired_edit_balanced": statistics.mean(paired_edit_means),
            "sample_self_agreement_paired_prompt_weighted": statistics.mean(prompt_weighted_paired),

            "sample_self_agreement_all_pairs_edit_balanced": statistics.mean(all_pair_edit_means),
            "sample_self_agreement_all_pairs_prompt_weighted": statistics.mean(prompt_weighted_all_pairs),

            "greedy_self_agreement_edit_balanced": (
                statistics.mean(greedy_edit_means)
                if greedy_edit_means else None
            ),
        }

    return result


selfloc = compute_selfloc_by_bucket(
    by_key0=by_key0,
    by_key1=by_key1,
    locality_buckets=["near_locality", "far_locality"],
)

overall_locality_edits = sum(item["num_edits"] for item in selfloc.values())

overall_selfloc = (
    sum(item[main_selfloc_key] * item["num_edits"] for item in selfloc.values())
    / overall_locality_edits
)


def extract_edit_balanced(summary, metric_name):
    return {
        bucket: values.get(metric_name)
        for bucket, values in summary.items()
    }


base_tfpr_by_bucket_seed0 = extract_edit_balanced(
    summary0,
    "mean_target_false_positive_rate_edit_balanced",
)

base_tfpr_by_bucket_seed1 = extract_edit_balanced(
    summary1,
    "mean_target_false_positive_rate_edit_balanced",
)

base_malformed_by_bucket_seed0 = extract_edit_balanced(
    summary0,
    "mean_malformed_rate_edit_balanced",
)

base_malformed_by_bucket_seed1 = extract_edit_balanced(
    summary1,
    "mean_malformed_rate_edit_balanced",
)


def mean_two_dicts(d0, d1):
    keys = sorted(set(d0) | set(d1))
    out = {}

    for key in keys:
        vals = [
            value for value in [d0.get(key), d1.get(key)]
            if value is not None
        ]
        out[key] = sum(vals) / len(vals) if vals else None

    return out


base_tfpr_by_bucket_mean = mean_two_dicts(
    base_tfpr_by_bucket_seed0,
    base_tfpr_by_bucket_seed1,
)

base_malformed_by_bucket_mean = mean_two_dicts(
    base_malformed_by_bucket_seed0,
    base_malformed_by_bucket_seed1,
)

report = {
    "protocol_version": "counterfact_direction1_v1",
    "split_role": "dev_tune_200",
    "balance_unit_field": UNIT_FIELD,
    "base_seed0_summary": summary0,
    "base_seed1_summary": summary1,
    "selfloc_by_bucket": selfloc,
    "main_selfloc_key": main_selfloc_key,
    "selfloc_base_near_far_overall": overall_selfloc,
    "base_TFPR_by_bucket_seed0": base_tfpr_by_bucket_seed0,
    "base_TFPR_by_bucket_seed1": base_tfpr_by_bucket_seed1,
    "base_TFPR_by_bucket_mean": base_tfpr_by_bucket_mean,
    "base_malformed_rate_by_bucket_seed0": base_malformed_by_bucket_seed0,
    "base_malformed_rate_by_bucket_seed1": base_malformed_by_bucket_seed1,
    "base_malformed_rate_by_bucket_mean": base_malformed_by_bucket_mean,
    "definition": (
        "Base/base agreement between seed0 and seed1 on near/far locality buckets. "
        "Rows are aligned at prompt-row level under edit_id, then averaged within edit_id, then across edits. "
        "The main denominator uses sample_self_agreement_paired_edit_balanced."
    ),
    "seed0_results": str(base0 / "per_case_results.jsonl"),
    "seed1_results": str(base1 / "per_case_results.jsonl"),
}

report_path = out_dir / "base_selfconsistency_summary.json"
with report_path.open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print()
print("Self-locality:")
for bucket, summary in selfloc.items():
    print(bucket, summary)
print("overall locality SelfLoc_base:", overall_selfloc)
print("base TFPR by bucket, mean:", base_tfpr_by_bucket_mean)
print("base malformed rate by bucket, mean:", base_malformed_by_bucket_mean)
print("Wrote:", report_path)


## Expected Outcome

You should get:

```text
Protocol config OK for both seed runs
matching prompt-row counts
edit-balanced base metrics by bucket
selfloc_by_bucket for near_locality and far_locality
rewrite/near/far num_edits = 200
base_selfconsistency_summary.json
```

Stop if a run config is missing, protocol fields are wrong, row counts differ, prompt rows cannot be aligned, or self-locality cannot be computed.

If this passes, Step 2B is complete. The next notebook should run the first real dev baseline comparison against base using the G0 and SelfLoc artifacts.
